<a href="https://colab.research.google.com/github/purabkumar/car/blob/main/sec_edgar_collector_structured.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:

import json
import hashlib
import time
from pathlib import Path
from datetime import datetime, timezone

import requests
import pandas as pd

COMPANY = {
    "name": "Apple Inc.",
    "ticker": "AAPL",
    "cik": "0000320193",
}

SEC_CONTACT_EMAIL = "admin@company.com"
COLLECTOR_VERSION = "1.0.0"
SOURCE_NAME = "SEC EDGAR"
ALLOWED_FORMS = {"10-K", "10-Q", "8-K"}

PROJECT_DIR = Path("financial_data")
RAW_DIR = PROJECT_DIR / "raw"
CLEAN_DIR = PROJECT_DIR / "clean"
VALIDATED_DIR = PROJECT_DIR / "validated"
STORE_DIR = PROJECT_DIR / "store"
for folder in [RAW_DIR, CLEAN_DIR, VALIDATED_DIR, STORE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

MASTER_FILE = STORE_DIR / "apple_filings_master.csv"
RUN_LOG_FILE = STORE_DIR / "run_log.csv"
REJECTED_FILE = VALIDATED_DIR / "apple_filings_rejected.csv"

if not SEC_CONTACT_EMAIL or SEC_CONTACT_EMAIL == "purabkumar479@gmail.com":
    raise ValueError("Please replace SEC_CONTACT_EMAIL with a real contact email before running.")

print("Configuration ready for", COMPANY["name"])
print("Project folder:", PROJECT_DIR.resolve())

Configuration ready for Apple Inc.
Project folder: /content/financial_data


In [29]:
# CELL 2 — Fetch the SEC submissions JSON
run_started = time.time()
run_time_utc = datetime.now(timezone.utc).isoformat()

cik_padded = COMPANY["cik"].zfill(10)
api_url = f"https://data.sec.gov/submissions/CIK{cik_padded}.json"
headers = {
    "User-Agent": f"FinancialDataIngestion/1.0 ({SEC_CONTACT_EMAIL})",
    "Accept-Encoding": "gzip, deflate",
    "Host": "data.sec.gov",
}

response = requests.get(api_url, headers=headers, timeout=30)
print("HTTP status:", response.status_code)
response.raise_for_status()

data = response.json()
print("SEC company name:", data.get("name"))
print("CIK:", data.get("cik"))
print("Entity type:", data.get("entityType"))

HTTP status: 200
SEC company name: Apple Inc.
CIK: 0000320193
Entity type: operating


In [30]:
# CELL 3 — Save the untouched raw response and its manifest
raw_fetched_at = datetime.now(timezone.utc)
raw_bytes = response.content
raw_hash = hashlib.sha256(raw_bytes).hexdigest()
raw_timestamp = raw_fetched_at.strftime("%Y%m%dT%H%M%SZ")
raw_file = RAW_DIR / f"sec_submissions_{COMPANY['ticker']}_{raw_timestamp}.json"
raw_file.write_bytes(raw_bytes)

manifest = {
    "source_name": SOURCE_NAME,
    "source_api_url": api_url,
    "company_name": data.get("name"),
    "ticker": COMPANY["ticker"],
    "cik": data.get("cik"),
    "fetched_at_utc": raw_fetched_at.isoformat(),
    "collector_version": COLLECTOR_VERSION,
    "raw_file": str(raw_file),
    "content_hash": raw_hash,
    "hash_algorithm": "SHA-256",
    "content_type": response.headers.get("Content-Type"),
    "http_status": response.status_code,
}
manifest_file = RAW_DIR / f"sec_submissions_{COMPANY['ticker']}_{raw_timestamp}.manifest.json"
manifest_file.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

# Verify the saved raw file is byte-for-byte identical to the response
saved_hash = hashlib.sha256(raw_file.read_bytes()).hexdigest()
assert saved_hash == raw_hash, "Raw response hash verification failed"
print("Raw response saved:", raw_file)
print("Manifest saved:", manifest_file)
print("Raw SHA-256 verified:", saved_hash)

Raw response saved: financial_data/raw/sec_submissions_AAPL_20261012T104641Z.json
Manifest saved: financial_data/raw/sec_submissions_AAPL_20261012T104641Z.manifest.json
Raw SHA-256 verified: 2821703cf04e1138923243a87cc5148673feb9d3792a5b6a7747d1e1a093bf7f


In [31]:
# CELL 4 — Build the clean filing metadata table
recent = data["filings"]["recent"]
filings_df = pd.DataFrame(recent)
filings_df = filings_df[filings_df["form"].isin(ALLOWED_FORMS)].copy()

needed_columns = ["form", "filingDate", "reportDate", "accessionNumber", "primaryDocument"]
filings_df = filings_df[needed_columns].copy()

clean_df = filings_df.copy()
for column in ["form", "accessionNumber", "primaryDocument"]:
    clean_df[column] = clean_df[column].astype("string").str.strip()

clean_df["filingDate"] = pd.to_datetime(clean_df["filingDate"], errors="coerce")
clean_df["reportDate"] = pd.to_datetime(clean_df["reportDate"], errors="coerce")
clean_df = clean_df.drop_duplicates().copy()
clean_df.insert(0, "cik", data.get("cik", cik_padded))
clean_df.insert(1, "ticker", COMPANY["ticker"])
clean_df.insert(2, "company_name", data.get("name", COMPANY["name"]))

clean_file = CLEAN_DIR / "apple_filings_clean.csv"
clean_df.to_csv(clean_file, index=False)

print("Clean records:", len(clean_df))
print("Filing type counts:")
print(clean_df["form"].value_counts())
display(clean_df.head(5))

Clean records: 146
Filing type counts:
form
8-K     102
10-Q     33
10-K     11
Name: count, dtype: Int64


,cik,ticker,company_name,form,filingDate,reportDate,accessionNumber,primaryDocument
27,0000320193,AAPL,Apple Inc.,10-Q,2026-07-31,2026-06-27,0000320193-26-000020,aapl-20260627.htm
28,0000320193,AAPL,Apple Inc.,8-K,2026-07-30,2026-07-30,0000320193-26-000018,aapl-20260730.htm
38,0000320193,AAPL,Apple Inc.,10-Q,2026-05-01,2026-03-28,0000320193-26-000013,aapl-20260328.htm
39,0000320193,AAPL,Apple Inc.,8-K,2026-04-30,2026-04-30,0000320193-26-000011,aapl-20260430.htm
43,0000320193,AAPL,Apple Inc.,8-K,2026-04-20,2026-04-17,0001140361-26-015711,ef20071035_8k.htm


In [32]:
# CELL 5 — Validate records and save valid/rejected outputs
validation_df = clean_df.copy()
required_columns = ["cik", "ticker", "company_name", "form", "filingDate", "accessionNumber", "primaryDocument"]
validation_errors = []

for _, row in validation_df.iterrows():
    errors = []
    for column in required_columns:
        value = row[column]
        if pd.isna(value) or str(value).strip() == "":
            errors.append(f"Missing required field: {column}")
    if pd.notna(row["form"]) and row["form"] not in ALLOWED_FORMS:
        errors.append("Unsupported filing type")
    if pd.isna(row["filingDate"]):
        errors.append("Invalid or missing filing date")
    if pd.notna(row["filingDate"]) and pd.notna(row["reportDate"]):
        if row["reportDate"] > row["filingDate"]:
            errors.append("Reporting date is after filing date")
    validation_errors.append(errors)

validation_df["validation_errors"] = validation_errors

# Flag repeated accession numbers after the first occurrence
duplicate_mask = validation_df["accessionNumber"].duplicated(keep="first")
validation_df.loc[duplicate_mask, "validation_errors"] = validation_df.loc[duplicate_mask, "validation_errors"].apply(
    lambda errors: errors + ["Duplicate accession number"]
)

valid_df = validation_df[validation_df["validation_errors"].str.len() == 0].copy()
rejected_df = validation_df[validation_df["validation_errors"].str.len() > 0].copy()
valid_df = valid_df.drop(columns=["validation_errors"])
rejected_df.to_csv(REJECTED_FILE, index=False)

validated_file = VALIDATED_DIR / "apple_filings_validated.csv"
valid_df.to_csv(validated_file, index=False)

print("Total clean records:", len(validation_df))
print("Valid records:", len(valid_df))
print("Rejected records:", len(rejected_df))
if len(rejected_df):
    display(rejected_df[["accessionNumber", "form", "validation_errors"]].head(20))

Total clean records: 146
Valid records: 146
Rejected records: 0


In [33]:
# CELL 6 — Add provenance and stable metadata hashes
current_df = valid_df.copy()
current_df["filingDate"] = current_df["filingDate"].dt.strftime("%Y-%m-%d")
current_df["reportDate"] = current_df["reportDate"].dt.strftime("%Y-%m-%d")

def make_filing_url(row):
    accession_without_dashes = str(row["accessionNumber"]).replace("-", "")
    cik_digits = str(row["cik"]).lstrip("0")
    return (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik_digits}/{accession_without_dashes}/{row['primaryDocument']}"
    )

def make_metadata_hash(row):
    # This hashes selected normalized metadata, not the full filing document.
    record = {
        "cik": str(row["cik"]),
        "accessionNumber": str(row["accessionNumber"]),
        "form": str(row["form"]),
        "filingDate": str(row["filingDate"]),
        "reportDate": "" if pd.isna(row["reportDate"]) else str(row["reportDate"]),
        "primaryDocument": str(row["primaryDocument"]),
    }
    payload = json.dumps(record, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

current_df["source_name"] = SOURCE_NAME
current_df["source_url"] = current_df.apply(make_filing_url, axis=1)
current_df["source_api_url"] = api_url
current_df["fetched_at_utc"] = run_time_utc
current_df["collector_version"] = COLLECTOR_VERSION
current_df["content_hash"] = current_df.apply(make_metadata_hash, axis=1)

print("Provenance fields added.")
display(current_df[["accessionNumber", "source_url", "fetched_at_utc", "content_hash"]].head(3))

Provenance fields added.


,accessionNumber,source_url,fetched_at_utc,content_hash
27,0000320193-26-000020,https://www.sec.gov/Archives/edgar/data/320193...,2026-10-12T10:46:41.402944+00:00,4435cccda8b0f199183d80cff3983a8326d4fcf9905920...
28,0000320193-26-000018,https://www.sec.gov/Archives/edgar/data/320193...,2026-10-12T10:46:41.402944+00:00,7cfba5359d0e6751d538337ef1614baa0ffd92f80348df...
38,0000320193-26-000013,https://www.sec.gov/Archives/edgar/data/320193...,2026-10-12T10:46:41.402944+00:00,1e6dde8be8eb6ffe730c2c435c3c34b5b0074c9ff0e190...


In [34]:
# CELL 7 — Idempotent master store: add only unseen accession numbers
if MASTER_FILE.exists():
    old_df = pd.read_csv(MASTER_FILE, dtype=str).fillna("")
    existing_ids = set(old_df["accessionNumber"].astype(str))
    new_df = current_df[~current_df["accessionNumber"].astype(str).isin(existing_ids)].copy()
    added_count = len(new_df)
    if added_count > 0:
        combined_df = pd.concat([old_df, new_df.astype(str)], ignore_index=True)
        combined_df.to_csv(MASTER_FILE, index=False)
    else:
        combined_df = old_df
else:
    new_df = current_df.copy()
    added_count = len(new_df)
    combined_df = new_df.astype(str)
    combined_df.to_csv(MASTER_FILE, index=False)

stored_df = pd.read_csv(MASTER_FILE, dtype=str).fillna("")
print("Records fetched:", len(current_df))
print("New records added this run:", added_count)
print("Total records stored:", len(stored_df))
print("Duplicate accession numbers in master:", stored_df["accessionNumber"].duplicated().sum())

Records fetched: 146
New records added this run: 0
Total records stored: 146
Duplicate accession numbers in master: 0


In [35]:
# CELL 8 — Save run report and final project summary
duration_seconds = round(time.time() - run_started, 2)
run_report = {
    "run_time_utc": run_time_utc,
    "source": SOURCE_NAME,
    "company": COMPANY["ticker"],
    "records_fetched": int(len(current_df)),
    "records_added": int(added_count),
    "records_rejected": int(len(rejected_df)),
    "duration_seconds": duration_seconds,
    "status": "SUCCESS",
}

if RUN_LOG_FILE.exists():
    log_df = pd.read_csv(RUN_LOG_FILE)
    log_df = pd.concat([log_df, pd.DataFrame([run_report])], ignore_index=True)
else:
    log_df = pd.DataFrame([run_report])
log_df.to_csv(RUN_LOG_FILE, index=False)

readme = f'''# SEC EDGAR Filings Collector — Prototype\n\n## Overview\nA Python/Google Colab prototype that collects Apple Inc. SEC filing metadata for 10-K, 10-Q, and 8-K filings.\n\n## Data flow\nSEC submissions API → raw JSON + manifest → clean CSV → validated CSV → master CSV + run log.\n\n## Current run\n- Records fetched: {len(current_df)}\n- Records added this run: {added_count}\n- Records rejected: {len(rejected_df)}\n- Total records stored: {len(stored_df)}\n- Duplicate accession numbers in master: {stored_df["accessionNumber"].duplicated().sum()}\n\n## Provenance\nRecords include source name, filing URL, API URL, UTC fetch time, collector version, and a SHA-256 hash of selected normalized metadata. The metadata hash is not a hash of the full filing document bytes.\n\n## Idempotency\nThe master store uses accession numbers to add only previously unseen filings. Existing records are left unchanged.\n\n## Limitations\nThis is a single-company prototype. It does not yet implement the full 20-company/S&P 500 scope, XBRL company facts, daily prices, RSS news, PostgreSQL, unattended scheduling, or comprehensive source-failure monitoring. Colab local files may be lost when the runtime is reset; use persistent storage for ongoing collection.\n\n## Run instructions\nOpen the notebook in Google Colab, set a valid SEC contact email in Cell 1, then run cells in order.\n'''
(PROJECT_DIR / "README.md").write_text(readme, encoding="utf-8")

print("RUN REPORT")
print(json.dumps(run_report, indent=2))
print("\nFINAL CHECKS")
print("Master file:", MASTER_FILE)
print("Run log:", RUN_LOG_FILE)
print("README:", PROJECT_DIR / "README.md")
print("Raw file:", raw_file)
print("Manifest:", manifest_file)
print("Unique accession numbers:", stored_df["accessionNumber"].nunique())
print("Master row count:", len(stored_df))
print("Run log rows:", len(log_df))

RUN REPORT
{
  "run_time_utc": "2026-10-12T10:46:41.402944+00:00",
  "source": "SEC EDGAR",
  "company": "AAPL",
  "records_fetched": 146,
  "records_added": 0,
  "records_rejected": 0,
  "duration_seconds": 0.28,
  "status": "SUCCESS"
}

FINAL CHECKS
Master file: financial_data/store/apple_filings_master.csv
Run log: financial_data/store/run_log.csv
README: financial_data/README.md
Raw file: financial_data/raw/sec_submissions_AAPL_20261012T104641Z.json
Manifest: financial_data/raw/sec_submissions_AAPL_20261012T104641Z.manifest.json
Unique accession numbers: 146
Master row count: 146
Run log rows: 4


In [36]:

from pathlib import Path
import zipfile

project_dir = Path("financial_data")
zip_path = Path("sec_edgar_collector_submission.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for file in project_dir.rglob("*"):
        if file.is_file():
            archive.write(file, file.as_posix())

print("Project packaged successfully:", zip_path)
print("ZIP size:", round(zip_path.stat().st_size / 1024, 2), "KB")

from google.colab import files
files.download(str(zip_path))


Project packaged successfully: sec_edgar_collector_submission.zip
ZIP size: 129.8 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>